## Health Dataset Generator

Generates a synthetic **health** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `health` | `patients` | ~5K | `medical_records` | 100K-500K | Comorbidity modeling (up to 3 conditions), ~35 ICD-10 codes, 50 physicians |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `health` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.health') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.health');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(71)
random.seed(71)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "health"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = date(2026, 3, 21)

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Patients table (~5,000 rows) ---
genders = ["Male", "Female"]
blood_types = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
blood_type_weights = [35.7, 6.3, 8.5, 1.5, 3.4, 0.6, 37.4, 6.6]
insurance_types = ["Private", "Medicare", "Medicaid", "Self-Pay", "Tricare"]
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West Coast", "Pacific Northwest"]

# 50 physicians generated with faker
physicians = [f"Dr. {fake.last_name()}" for _ in range(50)]

# All chronic conditions for comorbidity modeling
all_conditions = ["Hypertension", "Type 2 Diabetes", "Asthma", "COPD", "Heart Disease",
                  "Arthritis", "Depression", "Anxiety", "Chronic Kidney Disease",
                  "Hypothyroidism", "Osteoporosis", "Sleep Apnea", "GERD", "Obesity"]

# Comorbidity pairs (conditions that commonly co-occur)
comorbidity_pairs = {
    "Hypertension": ["Type 2 Diabetes", "Heart Disease", "Chronic Kidney Disease", "Obesity"],
    "Type 2 Diabetes": ["Hypertension", "Heart Disease", "Obesity", "Chronic Kidney Disease"],
    "COPD": ["Heart Disease", "Depression", "Hypertension", "Osteoporosis"],
    "Heart Disease": ["Hypertension", "Type 2 Diabetes", "Depression"],
    "Depression": ["Anxiety", "Obesity", "Sleep Apnea"],
    "Anxiety": ["Depression", "GERD", "Asthma"],
    "Obesity": ["Type 2 Diabetes", "Hypertension", "Sleep Apnea", "Arthritis"],
    "Arthritis": ["Osteoporosis", "Depression", "Obesity"],
}

NUM_PATIENTS = 5000
patients = []
for i in range(1, NUM_PATIENTS + 1):
    age = int(clamp(random.gauss(52, 18), 18, 95))
    gender = random.choice(genders)
    dob = NOW - timedelta(days=int(age * 365.25) + random.randint(0, 364))
    # Verify age is correctly derivable from DOB
    computed_age = (NOW - dob).days // 365
    bmi = round(clamp(random.gauss(27.5, 5.5), 16.0, 55.0), 1)
    is_smoker = random.choices([True, False], weights=[15, 85])[0]

    age_bp_offset = (age - 40) * 0.4
    bmi_bp_offset = max(0, (bmi - 25) * 0.8)
    systolic = int(clamp(random.gauss(125 + age_bp_offset + bmi_bp_offset, 18), 85, 200))
    diastolic = int(clamp(random.gauss(systolic * 0.6, 8), 50, 120))

    # Primary chronic condition with comorbidity support
    conditions = []
    if age >= 60:
        if random.random() < 0.40:
            if systolic > 140: conditions.append(random.choices(["Hypertension", "Heart Disease", "Chronic Kidney Disease"], weights=[50, 30, 20])[0])
            elif bmi > 30: conditions.append(random.choices(["Type 2 Diabetes", "Arthritis", "Heart Disease"], weights=[45, 35, 20])[0])
            elif is_smoker: conditions.append(random.choices(["COPD", "Heart Disease", "Hypertension"], weights=[45, 30, 25])[0])
            else: conditions.append(random.choices(["Arthritis", "Hypertension", "Depression", "Hypothyroidism"], weights=[30, 35, 20, 15])[0])
    elif age >= 40:
        if random.random() < 0.25:
            if bmi > 32: conditions.append(random.choices(["Type 2 Diabetes", "Hypertension", "Obesity"], weights=[40, 35, 25])[0])
            elif is_smoker: conditions.append(random.choices(["Asthma", "COPD", "Anxiety"], weights=[35, 30, 35])[0])
            else: conditions.append(random.choices(["Depression", "Anxiety", "Hypertension", "Asthma", "GERD"], weights=[25, 25, 20, 15, 15])[0])
    else:
        if random.random() < 0.12:
            conditions.append(random.choices(["Asthma", "Anxiety", "Depression", "GERD"], weights=[30, 30, 25, 15])[0])

    # Add comorbidity for patients with a primary condition (30-50% chance based on age)
    if conditions and random.random() < (0.30 + (age - 40) * 0.005):
        possible = comorbidity_pairs.get(conditions[0], [])
        if possible:
            second = random.choice(possible)
            if second not in conditions:
                conditions.append(second)
    # Rare triple comorbidity for elderly (10%)
    if len(conditions) == 2 and age >= 65 and random.random() < 0.10:
        possible = comorbidity_pairs.get(conditions[1], [])
        if possible:
            third = random.choice([c for c in possible if c not in conditions][:3]) if [c for c in possible if c not in conditions] else None
            if third:
                conditions.append(third)

    chronic_conditions = ", ".join(conditions) if conditions else None
    num_chronic = len(conditions)

    if age >= 65:
        ins = random.choices(insurance_types, weights=[20, 55, 10, 10, 5])[0]
    else:
        ins = random.choices(insurance_types, weights=[55, 5, 20, 15, 5])[0]

    patients.append(Row(
        patient_id=i,
        patient_name=fake.name(),
        date_of_birth=dob,
        age=computed_age,
        gender=gender,
        blood_type=random.choices(blood_types, weights=blood_type_weights)[0],
        insurance_type=ins,
        primary_physician=random.choice(physicians),
        region=random.choice(regions),
        bmi=bmi,
        systolic_bp=systolic,
        diastolic_bp=diastolic,
        is_smoker=is_smoker,
        chronic_conditions=chronic_conditions,
        num_chronic_conditions=num_chronic
    ))

patient_lookup = {p.patient_id: p for p in patients}

patients_df = spark.createDataFrame(patients)
patients_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.patients")
print(f"✔ Created {CATALOG_SCHEMA}.patients ({patients_df.count()} rows)")

# --- Medical Records table (randomized ~100K-500K rows) ---
visit_types = ["Routine Checkup", "Emergency", "Specialist Referral", "Urgent Care", "Telehealth", "Follow-Up"]

diag_dept_med = {
    "respiratory": [
        ("J06.9", "Acute upper respiratory infection", "Primary Care",    [("Acetaminophen", 30), ("Ibuprofen", 25), ("Amoxicillin", 30), (None, 15)]),
        ("J45.909", "Unspecified asthma",              "Pulmonology",     [("Albuterol", 60), ("Prednisone", 25), (None, 15)]),
        ("J20.9", "Acute bronchitis",                  "Primary Care",    [("Amoxicillin", 35), ("Albuterol", 25), ("Prednisone", 20), (None, 20)]),
        ("R05", "Cough",                               "Primary Care",    [("Amoxicillin", 20), (None, 50), ("Acetaminophen", 30)]),
        ("J02.9", "Acute pharyngitis",                 "Primary Care",    [("Amoxicillin", 45), ("Ibuprofen", 25), (None, 30)]),
        ("J18.9", "Pneumonia unspecified",              "Pulmonology",     [("Amoxicillin", 40), ("Azithromycin", 35), (None, 25)]),
        ("J44.1", "COPD with acute exacerbation",      "Pulmonology",     [("Albuterol", 40), ("Prednisone", 35), (None, 25)]),
    ],
    "cardiovascular": [
        ("I10", "Essential hypertension",              "Cardiology",      [("Lisinopril", 35), ("Amlodipine", 25), ("Losartan", 25), (None, 15)]),
        ("E78.5", "Hyperlipidemia",                    "Cardiology",      [("Atorvastatin", 60), ("Lisinopril", 15), (None, 25)]),
        ("I25.10", "Atherosclerotic heart disease",    "Cardiology",      [("Atorvastatin", 30), ("Aspirin", 30), ("Metoprolol", 25), (None, 15)]),
        ("I48.91", "Atrial fibrillation",              "Cardiology",      [("Warfarin", 35), ("Metoprolol", 30), ("Apixaban", 25), (None, 10)]),
    ],
    "endocrine": [
        ("E11.9", "Type 2 diabetes mellitus",          "Endocrinology",   [("Metformin", 65), ("Lisinopril", 10), (None, 25)]),
        ("E03.9", "Hypothyroidism unspecified",         "Endocrinology",   [("Levothyroxine", 70), (None, 30)]),
        ("E66.01", "Morbid obesity",                   "Endocrinology",   [("Metformin", 30), (None, 70)]),
    ],
    "musculoskeletal": [
        ("M54.5", "Low back pain",                     "Orthopedics",     [("Ibuprofen", 40), ("Gabapentin", 25), ("Acetaminophen", 20), (None, 15)]),
        ("M25.561", "Right knee pain",                 "Orthopedics",     [("Ibuprofen", 45), ("Acetaminophen", 25), (None, 30)]),
        ("M81.0", "Age-related osteoporosis",          "Orthopedics",     [("Alendronate", 50), ("Calcium", 30), (None, 20)]),
        ("M17.11", "Primary osteoarthritis right knee","Orthopedics",     [("Ibuprofen", 35), ("Acetaminophen", 30), ("Gabapentin", 20), (None, 15)]),
    ],
    "mental_health": [
        ("F41.1", "Generalized anxiety",               "Psychiatry",      [("Sertraline", 45), ("Gabapentin", 20), (None, 35)]),
        ("F32.9", "Major depressive disorder",         "Psychiatry",      [("Sertraline", 40), ("Fluoxetine", 25), ("Gabapentin", 10), (None, 25)]),
        ("F51.01", "Primary insomnia",                 "Psychiatry",      [("Trazodone", 45), ("Melatonin", 30), (None, 25)]),
    ],
    "gastrointestinal": [
        ("K21.0", "GERD",                              "Gastroenterology",[("Omeprazole", 65), (None, 35)]),
        ("R10.9", "Abdominal pain",                    "Gastroenterology",[("Omeprazole", 25), ("Acetaminophen", 20), (None, 55)]),
        ("K59.00", "Constipation",                     "Gastroenterology",[(None, 70), ("Omeprazole", 30)]),
        ("K58.9", "Irritable bowel syndrome",          "Gastroenterology",[("Omeprazole", 30), ("Dicyclomine", 35), (None, 35)]),
    ],
    "other": [
        ("N39.0", "Urinary tract infection",           "Primary Care",    [("Amoxicillin", 55), (None, 45)]),
        ("G43.909", "Migraine",                        "Neurology",       [("Ibuprofen", 30), ("Acetaminophen", 25), ("Gabapentin", 20), (None, 25)]),
        ("L30.9", "Dermatitis",                        "Dermatology",     [("Prednisone", 35), (None, 65)]),
        ("R51", "Headache",                            "Primary Care",    [("Ibuprofen", 35), ("Acetaminophen", 35), (None, 30)]),
        ("N18.3", "Chronic kidney disease stage 3",    "Nephrology",      [("Lisinopril", 40), ("Amlodipine", 25), (None, 35)]),
        ("G47.33", "Obstructive sleep apnea",          "Pulmonology",     [(None, 80), ("Modafinil", 20)]),
        ("E05.90", "Hyperthyroidism unspecified",      "Endocrinology",   [("Methimazole", 55), (None, 45)]),
    ],
}

all_diagnoses = []
for cat, diags in diag_dept_med.items():
    for d in diags:
        all_diagnoses.append((cat, *d))

condition_diag_weights = {
    "Hypertension":           {"cardiovascular": 40, "respiratory": 10, "endocrine": 10, "musculoskeletal": 10, "mental_health": 8, "gastrointestinal": 10, "other": 12},
    "Type 2 Diabetes":        {"cardiovascular": 15, "respiratory": 10, "endocrine": 35, "musculoskeletal": 10, "mental_health": 8, "gastrointestinal": 10, "other": 12},
    "Asthma":                 {"cardiovascular": 5,  "respiratory": 45, "endocrine": 5,  "musculoskeletal": 10, "mental_health": 10, "gastrointestinal": 10, "other": 15},
    "COPD":                   {"cardiovascular": 10, "respiratory": 45, "endocrine": 5,  "musculoskeletal": 10, "mental_health": 8, "gastrointestinal": 10, "other": 12},
    "Heart Disease":          {"cardiovascular": 45, "respiratory": 10, "endocrine": 10, "musculoskeletal": 5,  "mental_health": 8, "gastrointestinal": 10, "other": 12},
    "Arthritis":              {"cardiovascular": 8,  "respiratory": 10, "endocrine": 5,  "musculoskeletal": 45, "mental_health": 10, "gastrointestinal": 10, "other": 12},
    "Depression":             {"cardiovascular": 8,  "respiratory": 10, "endocrine": 5,  "musculoskeletal": 10, "mental_health": 45, "gastrointestinal": 10, "other": 12},
    "Anxiety":                {"cardiovascular": 8,  "respiratory": 10, "endocrine": 5,  "musculoskeletal": 10, "mental_health": 45, "gastrointestinal": 10, "other": 12},
    "Chronic Kidney Disease": {"cardiovascular": 25, "respiratory": 10, "endocrine": 15, "musculoskeletal": 10, "mental_health": 8, "gastrointestinal": 15, "other": 17},
    "Obesity":                {"cardiovascular": 15, "respiratory": 8,  "endocrine": 25, "musculoskeletal": 20, "mental_health": 12, "gastrointestinal": 10, "other": 10},
    "Hypothyroidism":         {"cardiovascular": 10, "respiratory": 10, "endocrine": 35, "musculoskeletal": 12, "mental_health": 15, "gastrointestinal": 8,  "other": 10},
    "Sleep Apnea":            {"cardiovascular": 20, "respiratory": 15, "endocrine": 10, "musculoskeletal": 10, "mental_health": 15, "gastrointestinal": 10, "other": 20},
    "GERD":                   {"cardiovascular": 8,  "respiratory": 12, "endocrine": 5,  "musculoskeletal": 10, "mental_health": 10, "gastrointestinal": 40, "other": 15},
}
default_diag_weights = {"cardiovascular": 12, "respiratory": 25, "endocrine": 8, "musculoskeletal": 15, "mental_health": 15, "gastrointestinal": 13, "other": 12}

patient_pool = []
for p in patients:
    visits_weight = 2 + p.num_chronic_conditions * 2
    if p.age > 65: visits_weight += 1
    patient_pool.extend([p.patient_id] * visits_weight)

visit_cost_params = {
    "Routine Checkup":      (5.0, 0.6, 75, 800),
    "Emergency":            (7.5, 1.0, 500, 75000),
    "Specialist Referral":  (5.8, 0.7, 100, 5000),
    "Urgent Care":          (5.3, 0.5, 100, 2000),
    "Telehealth":           (4.2, 0.4, 30, 300),
    "Follow-Up":            (4.6, 0.5, 50, 600),
}

records = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    visit_ts = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 730), hours=random.randint(7, 19), minutes=random.randint(0, 59))
    visit_type = random.choices(visit_types, weights=[30, 10, 20, 12, 15, 13])[0]

    pid = random.choice(patient_pool)
    patient = patient_lookup[pid]

    # Weighted diagnosis selection based on patient's chronic conditions
    primary_condition = patient.chronic_conditions.split(", ")[0] if patient.chronic_conditions else None
    diag_w = condition_diag_weights.get(primary_condition, default_diag_weights)
    cats = list(diag_w.keys())
    weights = list(diag_w.values())
    chosen_cat = random.choices(cats, weights=weights)[0]
    cat_diags = [d for d in all_diagnoses if d[0] == chosen_cat]
    chosen = random.choice(cat_diags)
    _, diag_code, diag_desc, dept, med_options = chosen

    if visit_type == "Emergency":
        dept = "Emergency"

    med_names = [m[0] for m in med_options]
    med_wts = [m[1] for m in med_options]
    medication = random.choices(med_names, weights=med_wts)[0]

    mu, sigma, lo, hi = visit_cost_params[visit_type]
    cost = float(round(clamp(random.lognormvariate(mu, sigma), lo, hi), 2))
    coverage_pct = round(clamp(random.betavariate(8.0, 3.0) * 100, 0, 100), 1)
    out_of_pocket = float(round(cost * (1 - coverage_pct / 100), 2))

    if visit_type == "Emergency" and random.random() < 0.35:
        los = int(clamp(random.expovariate(0.3), 1, 30))
    elif visit_type == "Specialist Referral" and random.random() < 0.1:
        los = int(clamp(random.expovariate(0.5), 1, 14))
    else:
        los = 0

    lab_prob = {"Routine Checkup": 0.6, "Emergency": 0.7, "Specialist Referral": 0.5,
                "Urgent Care": 0.3, "Telehealth": 0.05, "Follow-Up": 0.25}
    had_lab = random.random() < lab_prob.get(visit_type, 0.3)

    fu_prob = 0.35
    if patient.chronic_conditions: fu_prob = 0.50 + patient.num_chronic_conditions * 0.05
    if visit_type in ["Specialist Referral", "Emergency"]: fu_prob += 0.15
    follow_up = random.random() < min(fu_prob, 0.95)

    records.append(Row(
        record_id=30000 + i,
        patient_id=pid,
        visit_date=visit_ts,
        visit_type=visit_type,
        department=dept,
        diagnosis_code=diag_code,
        diagnosis_description=diag_desc,
        treatment_cost=cost,
        insurance_coverage_pct=coverage_pct,
        out_of_pocket_cost=out_of_pocket,
        length_of_stay_days=los,
        medication_prescribed=medication,
        lab_work_ordered=had_lab,
        follow_up_required=follow_up,
        physician=random.choice(physicians)
    ))

records_df = spark.createDataFrame(records)
records_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.medical_records")
print(f"✔ Created {CATALOG_SCHEMA}.medical_records ({records_df.count()} rows)")

print("\n--- Patients (sample) ---")
display(patients_df.limit(5))
print("\n--- Medical Records (sample) ---")
display(records_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.health.patients", {
    "patient_id":            "Unique identifier for the patient",
    "patient_name":          "Full name of the patient (generated via Faker)",
    "date_of_birth":         "Date of birth (DateType)",
    "age":                   "Current age in years, derived from date_of_birth",
    "gender":                "Gender: Male or Female",
    "blood_type":            "Blood type weighted to US population frequencies",
    "insurance_type":        "Insurance type: Private, Medicare, Medicaid, Self-Pay, or Tricare",
    "primary_physician":     "Assigned primary care physician (50 physicians, generated via Faker)",
    "region":                "Geographic region",
    "bmi":                   "Body Mass Index (normal distribution, mean 27.5)",
    "systolic_bp":           "Systolic blood pressure in mmHg (increases with age and BMI)",
    "diastolic_bp":          "Diastolic blood pressure in mmHg (correlated with systolic)",
    "is_smoker":             "Whether the patient is a current smoker (~15%)",
    "chronic_conditions":    "Comma-separated list of chronic conditions. Supports comorbidities (e.g. Hypertension, Type 2 Diabetes). NULL if healthy",
    "num_chronic_conditions":"Count of chronic conditions. 0 for healthy patients; up to 3 for elderly with comorbidities",
})

apply_comments(f"{CATALOG}.health.medical_records", {
    "record_id":              "Unique identifier for the medical visit record",
    "patient_id":             "Foreign key referencing patients.patient_id",
    "visit_date":             "Timestamp of the visit (TimestampType)",
    "visit_type":             "Type of visit: Routine Checkup, Emergency, Specialist Referral, Urgent Care, Telehealth, or Follow-Up",
    "department":             "Hospital department (~35 ICD-10 codes mapped to departments). Emergency overrides for ER visits",
    "diagnosis_code":         "ICD-10 diagnosis code (~35 codes across 7 categories)",
    "diagnosis_description":  "Human-readable diagnosis description",
    "treatment_cost":         "Total treatment cost in USD (log-normal per visit type)",
    "insurance_coverage_pct": "Percentage of cost covered by insurance (beta-distributed)",
    "out_of_pocket_cost":     "Patient responsibility in USD",
    "length_of_stay_days":    "Hospital stay in days. 0 for outpatient",
    "medication_prescribed":  "Name of prescribed medication; NULL if none",
    "lab_work_ordered":       "Whether lab work was ordered",
    "follow_up_required":     "Whether a follow-up visit is recommended. Higher for chronic patients",
    "physician":              "Attending physician for this visit",
})

print(f"\n\u2705 All column comments applied for health schema")

spark.sql(f"ALTER TABLE {CATALOG}.health.patients ALTER COLUMN patient_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.health.patients ADD CONSTRAINT pk_patients PRIMARY KEY (patient_id)")
spark.sql(f"ALTER TABLE {CATALOG}.health.medical_records ALTER COLUMN record_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.health.medical_records ADD CONSTRAINT pk_medical_records PRIMARY KEY (record_id)")
spark.sql(f"ALTER TABLE {CATALOG}.health.medical_records ADD CONSTRAINT fk_medical_records_patient_id FOREIGN KEY (patient_id) REFERENCES {CATALOG}.health.patients(patient_id)")
print(f"\u2714 PK/FK constraints applied for health schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.health') IS
'Health sample dataset with realistic statistical distributions and Faker-generated PII. Entity table: `patients` (~5K rows, PK: patient_id). Event table: `medical_records` (100K-500K rows, PK: record_id, FK: patient_id → patients). Key features: Comorbidity modeling (up to 3 conditions), ~35 ICD-10 codes, 50 physicians.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`health` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to health schema ({remove_after_value})")